In [8]:
import torch
from torch import nn


device = "cuda" if torch.cuda.is_available() else "cpu"


class SingleTokenMLP(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(paragraph_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2048, decoder_hidden_dim)
        )

    def forward(self, x):
        return self.net(x)


class EndToEndInverter(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim, prefix_len, decoder_model):
        super().__init__()
        self.prefix_len = prefix_len
        self.m_parallel_mlps = nn.ModuleList(
            [SingleTokenMLP(paragraph_dim, decoder_hidden_dim) for _ in range(prefix_len)]
        )
        self.decoder = decoder_model

    def _prefix(self, paragraph_embs):
        return torch.stack([mlp(paragraph_embs) for mlp in self.m_parallel_mlps], dim=1)

    @torch.no_grad()
    def generate(self, paragraph_embs, tokenizer, max_new_tokens=128):
        paragraph_embs = paragraph_embs.to(next(self.parameters()).device)
        prefix = self._prefix(paragraph_embs).to(dtype=self.decoder.dtype)
        outputs = self.decoder.generate(
            inputs_embeds=prefix,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.2,
        )
        return tokenizer.batch_decode(outputs, skip_special_tokens=True)


In [9]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
from peft import get_peft_model, LoraConfig, TaskType
import torch.nn.functional as F
import re

# Matches keys like:
#   decoder.base_model.model.model.layers.19.self_attn.q_proj.lora_A.default.weight
LORA_KEY_PATTERN = re.compile(r"\.([a-zA-Z0-9_]+)\.lora_(A|B)\.default\.weight$")


def detect_lora_config(state_dict):
    """
    Recovers target_modules and rank r directly from a checkpoint's state dict,
    instead of assuming a single hardcoded LoraConfig for every checkpoint.

    NOTE: lora_alpha cannot be recovered this way (it only scales the adapter
    output, it is never stored as a tensor). We default alpha = 2 * r, which
    matches the convention used in the training script, but if a given run
    used a different alpha this will silently use the wrong scale. Pass
    lora_alpha explicitly per-config below if you know the true value.
    """
    target_modules = set()
    ranks_seen = set()

    for key, tensor in state_dict.items():
        match = LORA_KEY_PATTERN.search(key)
        if match is None:
            continue
        proj_name, ab = match.group(1), match.group(2)
        target_modules.add(proj_name)
        # lora_A: [r, in_dim] -> r is dim 0 | lora_B: [out_dim, r] -> r is dim 1
        ranks_seen.add(tensor.shape[0] if ab == "A" else tensor.shape[1])

    if not target_modules:
        raise ValueError(
            "No LoRA weights found in this checkpoint (no '*.lora_A/B.default.weight' "
            "keys). Cannot auto-detect a LoRA config for it."
        )
    if len(ranks_seen) > 1:
        raise ValueError(f"Multiple LoRA ranks found in one checkpoint: {sorted(ranks_seen)}")

    return {"target_modules": sorted(target_modules), "r": ranks_seen.pop()}


class InverterFactory:
    def __init__(self):
        self.base_name = "Qwen/Qwen3-0.6B"

    def build_decoder(self, target_modules, r, lora_alpha=None):
        base = AutoModelForCausalLM.from_pretrained(
            self.base_name, device_map={"": 0}, trust_remote_code=True
        )

        alpha = lora_alpha if lora_alpha is not None else 2 * r

        lora = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=r,
            lora_alpha=alpha,
            lora_dropout=0.05,
            target_modules=target_modules,
        )

        print(f"  Building decoder with target_modules={target_modules}, r={r}, alpha={alpha}")
        return get_peft_model(base, lora)

    def load(self, repo, filename, prefix_len, paragraph_dim, lora_alpha=None):
        # Download and inspect the checkpoint FIRST, so the decoder we build
        # actually matches what's inside it, instead of guessing up front.
        state_dict = torch.load(
            hf_hub_download(repo_id=repo, filename=filename),
            map_location="cpu",
        )

        detected = detect_lora_config(state_dict)
        decoder = self.build_decoder(
            target_modules=detected["target_modules"],
            r=detected["r"],
            lora_alpha=lora_alpha,
        )

        model = EndToEndInverter(
            paragraph_dim=paragraph_dim,
            decoder_hidden_dim=decoder.config.hidden_size,
            prefix_len=prefix_len,
            decoder_model=decoder,
        )

        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print(repo, "missing:", len(missing), "unexpected:", len(unexpected))
        if missing or unexpected:
            print("  missing keys sample:", missing[:3])
            print("  unexpected keys sample:", unexpected[:3])

        return model.to(device).eval()

In [10]:
facts = [
    "Paris is the capital of France and it is known for its rich cultural heritage, world-renowned art museums, historic architecture, and famous landmarks like the Eiffel Tower and the Louvre Museum, which houses thousands of works including the Mona Lisa, making the city one of the most visited tourist destinations in the world and a global center for fashion, cuisine, and intellectual life, attracting millions of visitors every year who come to experience its vibrant street life, historical neighborhoods, iconic cafes, and its lasting influence on art, literature, and global culture throughout history",

    "The Earth revolves around the Sun once every year in an elliptical orbit, and this motion, along with its tilted axis, is responsible for the changing seasons we experience, influencing variations in temperature, daylight hours, and weather patterns across different regions of the planet throughout the year, while also playing a crucial role in sustaining life by regulating climate systems, supporting ecosystems, and enabling the natural cycles that govern agriculture, biodiversity, and environmental balance",

    "Water boils at one hundred degrees Celsius under standard atmospheric pressure, and this physical property is widely used in cooking, scientific experiments, and industrial processes, while also serving as a reference point in temperature measurement systems and changing slightly with variations in altitude and pressure, making it an essential concept in thermodynamics and practical applications where precise temperature control and understanding of phase changes are critical for efficiency and safety",

    "The human brain controls the entire body by sending electrical and chemical signals through the nervous system, allowing us to think, move, feel emotions, process sensory information, store memories, make decisions, and respond effectively to our environment in both conscious and unconscious ways, while also coordinating complex bodily functions such as breathing, heartbeat, and hormonal regulation, making it one of the most sophisticated and vital organs in the human body",

    "Mount Everest is the highest mountain above sea level, standing at over 8800 meters, and it is located in the Himalayas on the border between Nepal and China, attracting climbers from around the world despite its extreme conditions, including low oxygen levels, freezing temperatures, and challenging terrain, and it remains a symbol of human endurance and exploration as expeditions continue to push the limits of physical and mental capability in one of the harshest environments on Earth"
]

In [11]:
!uv pip install rouge_score nltk absl-py evaluate

Using Python 3.12.13 environment at: /usr
Checked 4 packages in 124ms


In [12]:
class Encoder:
    def __init__(self, model_name="Qwen/Qwen3-Embedding-0.6B"):
        self.model = AutoModel.from_pretrained(model_name).to(device).eval()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def encode(self, texts):
        inputs = self.tokenizer(
            texts, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = self.model(**inputs)

        token_embeddings = outputs.last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        sentence_embeddings = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)
        return F.normalize(sentence_embeddings, p=2, dim=1)

In [13]:
import evaluate
import numpy as np
import torch.nn.functional as F

class Evaluator:
    def __init__(self, encoder):
        print("Loading evaluation metrics (this might download some small models)...")
        self.rouge = evaluate.load('rouge')
        self.bleu = evaluate.load('bleu')
        self.meteor = evaluate.load('meteor')
        self.bertscore = evaluate.load('bertscore')
        self.encoder = encoder  # <-- We now give the evaluator access to the encoder!

    def score(self, inputs, outputs):
        # 1. Standard Metrics
        r_score = self.rouge.compute(predictions=outputs, references=inputs)
        b_score = self.bleu.compute(predictions=outputs, references=inputs)
        m_score = self.meteor.compute(predictions=outputs, references=inputs)
        bs_score = self.bertscore.compute(predictions=outputs, references=inputs, lang="en")
        
        # 2. Embedding Cosine Similarity
        # Encode both the original text and the generated text
        input_embs = self.encoder.encode(inputs)
        output_embs = self.encoder.encode(outputs)
        
        # Calculate how mathematically identical they are (1.0 is perfect)
        cos_sims = F.cosine_similarity(input_embs, output_embs, dim=1).cpu().numpy()
        
        return {
            "ROUGE-1": r_score['rouge1'],
            "ROUGE-2": r_score['rouge2'],
            "ROUGE-L": r_score['rougeL'],
            "BLEU": b_score['bleu'],
            "METEOR": m_score['meteor'],
            "BERTScore-F1": np.mean(bs_score['f1']),
            "Cosine-Similarity": np.mean(cos_sims)  
        }

In [14]:
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 79.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.2

In [15]:
class Runner:
    def __init__(self):
        self.encoder = Encoder()
        self.factory = InverterFactory()
        self.evaluator = Evaluator(self.encoder) # <-- Passed the encoder here!

    def run(self, configs, texts):
        embeddings = self.encoder.encode(texts)
        tokenizer = self.encoder.tokenizer

        results = {}

        for cfg in configs:
            model = self.factory.load(
                repo=cfg["repo"],
                filename=cfg["filename"],
                prefix_len=cfg["prefix_len"],
                paragraph_dim=embeddings.shape[1],
            )

            outputs = model.generate(embeddings, tokenizer, max_new_tokens=128)
            
            scores = self.evaluator.score(texts, outputs)

            results[cfg["repo"]] = {"scores": scores, "output": outputs}

            del model
            torch.cuda.empty_cache()

        return results

    
    def report(self, texts, results):
        names = list(results.keys())
    
        for i in range(len(texts)):
            print(f"\n{'='*80}")
            print(f"=== Sample {i+1} ===")
            print("ORIGINAL Input:", texts[i])
    
            for n in names:
                print(f"\n--- Model: {n} ---")
                print("Generated Output:", results[n]["output"][i])
                print("\nMetrics for this sample:")
                
                # Calculate scores for just this single sample
                single_score = self.evaluator.score([texts[i]], [results[n]["output"][i]])
                for k, v in single_score.items():
                    val = v.item() if hasattr(v, "item") else v
                    print(f"  {k}: {val:.4f}")
                    
        # Print Overall Averages
        print(f"\n{'='*80}")
        print("OVERALL AVERAGE SCORES (Across all samples)")
        print(f"{'='*80}")
        for n in names:
            print(f"\nModel: {n}")
            for k, v in results[n]["scores"].items():
                val = v.item() if hasattr(v, "item") else v
                print(f"  Avg {k}: {val:.4f}")
   



In [16]:
configs = [
    {
        "repo": "jg-eno/ReLoDer_v1",
        "prefix_len": 64,
        "filename": "best_checkpoint.pt"
    },
    {
        "repo": "jg-eno/ReLoDer_v2",
        "prefix_len": 64,
        "filename": "best_steps_checkpoint.pt"
    },
]

# 1. Create the Runner (This triggers the "Loading evaluation metrics..." print)
runner = Runner()

# 2. Run the models and get the results
results = runner.run(configs, facts)

# 3. Print the beautiful report!
runner.report(facts, results)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


best_steps_checkpoint.pt:   0%|          | 0.00/1.81G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  Building decoder with target_modules=['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj'], r=32, alpha=64
jg-eno/ReLoDer_v2 missing: 0 unexpected: 0

=== Sample 1 ===
ORIGINAL Input: Paris is the capital of France and it is known for its rich cultural heritage, world-renowned art museums, historic architecture, and famous landmarks like the Eiffel Tower and the Louvre Museum, which houses thousands of works including the Mona Lisa, making the city one of the most visited tourist destinations in the world and a global center for fashion, cuisine, and intellectual life, attracting millions of visitors every year who come to experience its vibrant street life, historical neighborhoods, iconic cafes, and its lasting influence on art, literature, and global culture throughout history

--- Model: jg-eno/ReLoDer_v1 ---
Generated Output: Paris is known for its rich history, art and literature, music, fashion, architecture, tourism, shopping, cuisine and many other cu